# Lecture 03, appendix: Neural Networks with PyTorch

**This notebook is optional.** The project does not need a neural network, and nothing in it is part of the examination. It belongs to the appendix of the lecture 3 slides, for those who want to see how a neural network works.

It trains a small network on the project data, measures it with five-fold cross-validation and searches its optimal hyperparameters. It uses the same cleaned data and the same split as notebook 02.

In [ ]:
!uv pip install pandas matplotlib scikit-learn torch

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch import nn

BLUE, PINK, SLATE, GREY = "#77C5D8", "#E5007D", "#466873", "#9A9A9A"
SEED = 42

df = pd.read_csv("../../data/credit_risk_dataset.csv").drop_duplicates()
late_cols = ["late_30_59", "late_60_89", "late_90"]
df = df[~df[late_cols].isin([96, 98]).any(axis=1)]

X, y = df.drop(columns="dlq_2yrs"), df["dlq_2yrs"]
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)
print(len(X_train), len(X_val), len(X_test))

## 1. Tensors and scaled inputs

PyTorch computes on **tensors**. A tensor is an array like a NumPy array. It can also live on a GPU, and it remembers how it was computed, which is what makes the gradients of section 3 possible.

A network learns with gradients, so its inputs must be on a similar scale. We clip the extreme values at the 1st and 99th percentile and standardise, as in notebook 01. Both are learned on the training set only.

In [ ]:
def fit_scaling(X):
    """Clip limits, means and standard deviations, learned from the rows passed in."""
    low, high = np.quantile(X, [0.01, 0.99], axis=0)
    clipped = np.clip(X, low, high)
    return low, high, clipped.mean(axis=0), clipped.std(axis=0)


def scale(X, scaling):
    low, high, mean, std = scaling
    return torch.tensor((np.clip(X, low, high) - mean) / std, dtype=torch.float32)


def as_tensor(labels):
    return torch.tensor(labels.to_numpy(), dtype=torch.float32)


scaling = fit_scaling(X_train.to_numpy())
Xt_train, Xt_val = scale(X_train.to_numpy(), scaling), scale(X_val.to_numpy(), scaling)
yt_train, yt_val = as_tensor(y_train), as_tensor(y_val)
print(Xt_train.shape, Xt_train.dtype)

## 2. From one neuron to a network

A **neuron** adds up its weighted inputs, $z = w_1 x_1 + \dots + w_n x_n + b$, and passes $z$ through an **activation**. Logistic regression is a single neuron with the sigmoid as its activation.

A **network** stacks layers of neurons. The network below has 10 inputs, one hidden layer of 32 neurons with the ReLU activation, $\mathrm{ReLU}(z) = \max(0, z)$, and one output neuron. The output is a score $z$, also called a **logit**. The sigmoid turns it into the probability of default.

In [ ]:
torch.manual_seed(SEED)
model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 1))
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

The 385 parameters are $10 \cdot 32$ weights and 32 biases in the hidden layer, plus 32 weights and one bias in the output neuron. The next cell computes one applicant by hand, with the formulas from the slides, and compares the result with the model.

In [ ]:
x = Xt_val[:1]
hidden, relu, output = model
h = torch.relu(x @ hidden.weight.T + hidden.bias)
z = h @ output.weight.T + output.bias
print(f"by hand: z = {z.item():.4f}   model: z = {model(x).item():.4f}   p = {torch.sigmoid(z).item():.3f}")

## 3. Loss, gradient descent and backpropagation

The loss is the log-loss from notebook 02. `nn.BCEWithLogitsLoss` applies the sigmoid and the log-loss in one numerically stable step.

Gradient descent moves every weight a small step against its gradient, $w \leftarrow w - \eta \, \partial L / \partial w$. The learning rate $\eta$ sets the size of the step. The cell below runs gradient descent by hand on the loss $L(w) = (w - 2)^2 + 1$, whose minimum lies at $w = 2$, with three learning rates.

In [ ]:
def descend(learning_rate, w=-1.0, steps=12):
    path = [w]
    for _ in range(steps):
        w = w - learning_rate * 2 * (w - 2)
        path.append(w)
    return np.array(path)

grid = np.linspace(-2, 6, 200)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for ax, (rate, label) in zip(axes, [(0.05, "too small"), (0.3, "about right"), (0.95, "too large")]):
    path = descend(rate)
    ax.plot(grid, (grid - 2) ** 2 + 1, color=GREY)
    ax.plot(path, (path - 2) ** 2 + 1, color=PINK, marker="o", markersize=4)
    ax.set_title(f"learning rate {rate}: {label}")
    ax.set_xlabel("weight w")
axes[0].set_ylabel("loss L(w)")
plt.tight_layout()
plt.show()

A network has hundreds of weights, and the loss depends on each of them through several layers. **Backpropagation** applies the chain rule from the loss back to every weight. PyTorch does it for you in `loss.backward()`.

For the bias of the output neuron the chain rule gives a short result. The derivative of the log-loss by $z$ is $\hat p - y$, so the gradient of the bias is the mean of $\hat p - y$ over all rows. Compare it with what PyTorch computes.

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
logits = model(Xt_train).squeeze(1)
loss = loss_fn(logits, yt_train)
loss.backward()

p = torch.sigmoid(logits)
print(f"log-loss before training: {loss.item():.4f}")
print(f"gradient of the output bias: autograd {output.bias.grad.item():.5f}, by hand {(p - yt_train).mean().item():.5f}")

## 4. The training loop

One **epoch** passes over all training rows once, in **mini-batches** of 256 rows. Each mini-batch is one step of the loop: predict, measure the loss, compute the gradients, update the weights. The optimiser Adam adapts the step size of every weight on its own, which makes it less sensitive to the learning rate than plain gradient descent.

After every epoch the function measures the loss on the validation set. **Early stopping** ends the training when that loss has not improved for 10 epochs, and keeps the weights of the best epoch. The function takes its settings from a `config`, as in notebook 03.

In [ ]:
def build(hidden=32, layers=1, dropout=0.0):
    blocks, width = [], 10
    for _ in range(layers):
        blocks += [nn.Linear(width, hidden), nn.ReLU(), nn.Dropout(dropout)]
        width = hidden
    return nn.Sequential(*blocks, nn.Linear(width, 1))


def train(config, X_fit, y_fit, X_check, y_check, max_epochs=100, patience=10, device="cpu"):
    torch.manual_seed(SEED)
    model = build(config["hidden"], config["layers"], config["dropout"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    loss_fn = nn.BCEWithLogitsLoss()
    batches = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_fit, y_fit), batch_size=config["batch_size"],
                                          shuffle=True, generator=torch.Generator().manual_seed(SEED))
    X_fit, y_fit, X_check, y_check = X_fit.to(device), y_fit.to(device), X_check.to(device), y_check.to(device)

    history, best_loss, best_weights, waited = [], float("inf"), None, 0
    for epoch in range(max_epochs):
        model.train()
        for X_batch, y_batch in batches:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()                                        # forget the last gradients
            loss = loss_fn(model(X_batch).squeeze(1), y_batch)           # 1 predict, 2 measure the loss
            loss.backward()                                              # 3 compute the gradients
            optimizer.step()                                             # 4 update the weights
        model.eval()
        with torch.no_grad():
            history.append((loss_fn(model(X_fit).squeeze(1), y_fit).item(),
                            loss_fn(model(X_check).squeeze(1), y_check).item()))
        if history[-1][1] < best_loss:
            best_loss, best_weights, waited = history[-1][1], {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            waited += 1
            if waited == patience:
                break
    model.load_state_dict(best_weights)
    return model, np.array(history)


def predict(model, X):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(X.to(next(model.parameters()).device)).squeeze(1)).cpu().numpy()

In [ ]:
config = {"hidden": 32, "layers": 1, "dropout": 0.0, "lr": 0.001, "weight_decay": 0.0, "batch_size": 256}

start = time.perf_counter()
model, history = train(config, Xt_train, yt_train, Xt_val, yt_val)
best_epoch = int(history[:, 1].argmin())
print(f"{len(history)} epochs in {time.perf_counter() - start:.1f} s, best epoch {best_epoch + 1}")
print(f"validation AUC: {roc_auc_score(y_val, predict(model, Xt_val)):.3f}")

plt.plot(history[:, 0], color=BLUE, label="training loss")
plt.plot(history[:, 1], color=PINK, ls="--", label="validation loss")
plt.axvline(best_epoch, color=GREY, ls=":", label=f"best epoch {best_epoch + 1}")
plt.xlabel("epoch"); plt.ylabel("log-loss"); plt.legend(frameon=False)
plt.show()

## 5. Cross-validation

One validation score is one draw, as section 9 of notebook 02 showed. Five-fold cross-validation trains the network five times on four fifths of the training set and scores it on the fifth. The scaling is learned again inside every fold, on its four fifths only.

ROC AUC serves as the example metric, as in notebook 03. Early stopping watches the held-out fifth, so the score is slightly optimistic. The final comparison in section 7 uses the separate validation set.

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)


def cross_validate(config):
    scores = []
    for fit_rows, held_out in folds.split(X_train, y_train):
        X_fit, X_held = X_train.iloc[fit_rows].to_numpy(), X_train.iloc[held_out].to_numpy()
        fold_scaling = fit_scaling(X_fit)
        model, _ = train(config, scale(X_fit, fold_scaling), as_tensor(y_train.iloc[fit_rows]),
                         scale(X_held, fold_scaling), as_tensor(y_train.iloc[held_out]))
        scores.append(roc_auc_score(y_train.iloc[held_out], predict(model, scale(X_held, fold_scaling))))
    return np.mean(scores), np.std(scores, ddof=1)


mean, std = cross_validate(config)
print(f"5-fold AUC: {mean:.3f} ± {std:.3f}")

## 6. A hyperparameter search

The settings of a network are its **hyperparameters**: the number of hidden layers and neurons, the learning rate, the batch size, and two ways to fight overfitting. **Dropout** switches off a random share of the neurons in every training step. **Weight decay** pulls every weight a little towards zero.

A random search draws 12 combinations and cross-validates each. It takes a few minutes on a laptop. With W&B, the same search runs as a sweep, as in notebook 03.

In [ ]:
search_space = {"hidden": [16, 32, 64, 128], "layers": [1, 2], "dropout": [0.0, 0.2],
                "weight_decay": [0.0, 1e-4, 1e-3], "batch_size": [64, 256]}
rng = np.random.default_rng(SEED)

trials = []
for trial in range(12):
    config = {name: rng.choice(values).item() for name, values in search_space.items()}
    config["lr"] = round(float(10 ** rng.uniform(-4, -2)), 5)
    mean, std = cross_validate(config)
    trials.append({**config, "auc": mean, "std": std})
    print(f"trial {trial + 1:2d}: AUC {mean:.3f} ± {std:.3f}   {config}")

results = pd.DataFrame(trials).sort_values("auc", ascending=False)
results.round(4)

In [ ]:
ranked = results.reset_index(drop=True)
plt.errorbar(ranked.index + 1, ranked["auc"], yerr=ranked["std"], fmt="o", color=SLATE, ecolor=GREY, capsize=3)
plt.scatter([1], ranked["auc"].iloc[:1], color=PINK, zorder=3, label="best combination")
plt.xlabel("combination, ranked"); plt.ylabel("5-fold AUC, mean ± std"); plt.legend(frameon=False)
plt.show()

The best combinations differ by less than their standard deviation. The data cannot tell them apart, which is typical for a small tabular dataset.

## 7. The best network against logistic regression

The best combination is trained once more on the whole training set, with early stopping on the validation set. Logistic regression gets the same clipped and standardised inputs.

In [ ]:
best = {name: ranked.loc[0, name] for name in [*search_space, "lr"]}
best = {name: value.item() if hasattr(value, "item") else value for name, value in best.items()}
network, _ = train(best, Xt_train, yt_train, Xt_val, yt_val)

logreg = LogisticRegression(max_iter=1000).fit(Xt_train.numpy(), y_train)

print("best combination:", best)
pd.Series({"logistic regression": roc_auc_score(y_val, logreg.predict_proba(Xt_val.numpy())[:, 1]),
           "neural network": roc_auc_score(y_val, predict(network, Xt_val))}, name="validation AUC").round(3)

On tabular data like ours the network beats logistic regression only by a small margin, and it takes far more effort to train. Networks pay off with images, sound and text, where there are many inputs and the structure in them matters.

## 8. CPU or GPU?

A GPU runs thousands of small calculations in parallel. It pays off for large matrix products, which are the core of big networks. For a small network, copying the data to the GPU and starting its work costs more than the GPU saves.

PyTorch runs on the CPU unless you move the model and the data to another device with `.to(device)`. `cuda` is an NVIDIA GPU and `mps` is the GPU of an Apple Silicon Mac. The cell compares both on your machine.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("accelerator found:", device)


def time_matrix_product(device, n=4096, repeats=10):
    a = torch.randn(n, n, device=device)
    (a @ a).sum().item()                       # warm-up
    start = time.perf_counter()
    for _ in range(repeats):
        b = a @ a
    b.sum().item()                             # waits until the device has finished
    return (time.perf_counter() - start) / repeats


for name in dict.fromkeys(["cpu", device]):
    start = time.perf_counter()
    train(best, Xt_train, yt_train, Xt_val, yt_val, max_epochs=10, patience=10, device=name)
    epochs = time.perf_counter() - start
    print(f"{name:>4}: 10 epochs of our network {epochs:.2f} s, one 4096 x 4096 matrix product {time_matrix_product(name) * 1000:.0f} ms")

If your machine has no GPU, both lines show the CPU. Hosted notebooks such as Google Colab offer a GPU for free, if you want to try one. Our network does not need one.